# Point-in-Time Quantitative Research Demo — Financial Data Lake

This sample research notebook demonstrates how a quantitative researcher queries the Financial Data Lake **without lookahead bias**:
1. Initialize a local lake instance and ingest manifest-verified synthetic fixtures.
2. Inspect the catalog, licenses, and entitlements.
3. Query fundamentals **as-of** two different dates to observe point-in-time restatement isolation (excluding future revisions).
4. Query split- and dividend-adjusted EOD prices before and after corporate action availability.
5. Pin and reproduce an immutable research snapshot.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path
import tempfile

from financial_data_lake import AccessContext, FinancialDataLake
from financial_data_lake.query import compute_rows_digest

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
tmp_dir = Path(tempfile.mkdtemp(prefix='fdl_notebook_'))

lake = FinancialDataLake(
    lake_root=tmp_dir,
    config_path=repo_root / 'config' / 'lake_config.json',
    entitlements_path=repo_root / 'config' / 'entitlements.json',
    schemas_path=repo_root / 'config' / 'schemas' / 'registry.json',
)

eng_ctx = AccessContext(principal='notebook-ingest', role='data_engineer', purpose='ingest')
res_ctx = AccessContext(principal='quant-alice', role='quant_research', purpose='research')

ingest_results = lake.ingest_all_fixtures(repo_root / 'data' / 'fixtures', eng_ctx)
for r in ingest_results:
    print(f'{r.dataset_id:22s} | source={r.source_name:28s} | curated={r.curated_rows} | deduped={r.deduplicated_rows}')

## 1. Point-in-Time Fundamentals (Excluding Future Restatements)

On `2024-06-07T23:59:59Z`, AAPL's 2024Q1 restatement (filed `2024-06-10T21:00:00Z`) is **not yet available**. An as-of query must return Revision 1 (`90.753B` revenue, `$1.53` EPS). After `2024-06-10T21:00:00Z`, Revision 2 (`90.120B`, `$1.51`) becomes visible.

In [ ]:
t_before_restatement = datetime(2024, 6, 7, 23, 59, 59, tzinfo=UTC)
t_after_restatement = datetime(2024, 6, 11, 0, 0, 0, tzinfo=UTC)

before = lake.query_as_of('fundamentals', t_before_restatement, res_ctx, filters={'instrument_id': 'INST-AAPL'})
after = lake.query_as_of('fundamentals', t_after_restatement, res_ctx, filters={'instrument_id': 'INST-AAPL'})

print('As-of 2024-06-07:', [(r['metric_name'], r['metric_value'], f"rev={r['revision_number']}") for r in before])
print('As-of 2024-06-11:', [(r['metric_name'], r['metric_value'], f"rev={r['revision_number']}") for r in after])

## 2. Point-in-Time Split & Dividend Adjusted Prices

AAPL underwent a 2-for-1 split with `ex_date=2024-06-05` (`available_at=2024-06-04T22:00:00Z`).
Querying as-of `2024-06-04T21:00:00Z` does NOT leak the split; querying as-of `2024-06-06T23:59:59Z` backward-adjusts pre-split closes from `200.0` to `100.0` and doubles pre-split volumes.

In [ ]:
adj_before = lake.query_adjusted_prices_as_of(datetime(2024, 6, 4, 21, 0, 0, tzinfo=UTC), res_ctx, symbol='AAPL')
adj_after = lake.query_adjusted_prices_as_of(datetime(2024, 6, 6, 23, 59, 59, tzinfo=UTC), res_ctx, symbol='AAPL')

print('Before split availability:')
for r in adj_before:
    print(f"  {r['trade_date']} raw_close={r['close']} adj_close={r['adj_close']} adj_vol={r['adj_volume']}")

print('After split availability:')
for r in adj_after:
    print(f"  {r['trade_date']} raw_close={r['close']} adj_close={r['adj_close']} adj_vol={r['adj_volume']}")

## 3. Reproducible Research Snapshot

Pin an immutable snapshot manifest across `prices_eod`, `corporate_actions`, and `fundamentals` so any future re-run reproduces identical rows and SHA-256 content digests.

In [ ]:
snap = lake.create_snapshot(
    name='q2-research-baseline',
    datasets=['prices_eod', 'corporate_actions', 'fundamentals'],
    as_of_ts=datetime(2024, 6, 7, 23, 59, 59, tzinfo=UTC),
    access=res_ctx,
)
pinned_fundamentals = lake.query_snapshot('q2-research-baseline', 'fundamentals', res_ctx)
print('Snapshot ID:', snap.snapshot_id)
print('Manifest SHA-256:', snap.manifest_sha256)
print('Pinned fundamentals row digest:', compute_rows_digest(pinned_fundamentals))